# AP 155 Lab Exercise
---
## Exercise 5: Matrices

_Instructions_: 
- **Report figure:** No report figure. Points in the report figure will be allocated to the report discussion 
- **Report discussion:**
  - What were the basis linear equations used to set up the matrix in each question of Problem 2?
  - Once you set up the matrices, what function did you use to solve the matrix equation (Could have different answers)? Check the docstring/documentation/other references and summarize the mechanism behind the function 
- **Code**: Complete the code and ensure it is functional, error-free, readable, and efficient (where needed). Include concise Markdown documentation highlighting the critical steps of your algorithm for each problem.

### Student Information
- _Full Name (Last Name, First Name)_: Domingo, Judiele Fatima P. 
- _Student No._: 2023-05167
- _Section_: THU-TX-1

### Grading Information (c/o Lab Instructor)
- [Rubrics description link](https://drive.google.com/file/d/1BMSlPot2Mc7XLu0eo4S8I8gLBsIadbCL/view?usp=sharing) (Note: percentages may still be tweaked)

| Criteria | Score | Subtotal |
| --- | --- | --- |
| Report figure | XX | 20 |
| Report discussion | XX | 20|
| Code readability | XX | 20 |
| Code efficiency | XX | 20 | 
| Code appropriateness  | XX | 20 | 
| **TOTAL** | XXX | 100 |

_Date and Time Scored (MM/DD/YYYY HH:MM AM/PM):_:_

---
## Section 1: Report

<!-- ### Report figure

<img src="figures/cat.jpg" alt="Alt text" width=45%> <img src="figures/cat.jpg" alt="Alt text" width=45%>


EDIT ME. See instruction above for what to include. Insert caption here, a short description what the figure illustrates. The syntax in markdown for putting a figure is `![Description](local_file_name.png)` or `<img src="/path/to_image.jpg" alt="Alt text" width=45%>`. Make sure the figure has complete elements.
-->

### Report discussion

For problem 2 part 1, the basis linear equations were derived from the static equilibrium condition ($\sum F = 0$) with $M$ as the hanging mass, $T_1$ as the tension of rope with corresponding angle $\alpha$, and $T_2$ corresponding to angle $\beta$. The sum of forces along the x-axis is given by $-T_1\cos(\alpha)+T_2\cos(\beta)=0$. While the equilibrium equation for the y-axis is given by $T_1\sin(\alpha)+T_2sin(\beta)=Mg$, g is gravity. From these equations, $A\vec{v}=\vec{w}$ was derived where A is the coefficient matrix, $\vec{v}$ contains T1 and T2, and $\vec{w}$contains 0 and $Mg$, the results of the basis equations The following matrix equation was used:

$\begin{pmatrix} -\cos(\alpha) & cos(\beta) \\ \sin(\alpha) &\sin(\beta) \end{pmatrix}$ $\begin{pmatrix} T_1 \\ T_2 \end{pmatrix}$ $= \begin{pmatrix} 0 \\ Mg \end{pmatrix}$

2 methods were used to solve the matrix equation. Firstly, np.linalg.solve was used; the function decomposes the coefficient matrix, A, into a lower and upper triangular matrices and solves the system by forward and backward substitution. The second method uses np.linalg.inv together with np.dot. The funtion np.linalg.inv first performs LU decomposition on A then performs forward and backward substitutions N times and stitches the N column vectors together to get $A^{-1}$. Then np.dot is applied which iterates through each row of $A^{-1}$ and gets the sum of the products of each $A^{-1}$ row element to each element of vector matrix \vec{w}. This method uses multiple iterations and loops and an additional matrix multiplication operation. The obtained results were similar however, processing time increased for the np.linalg.inv method. 

### Other comments
- N/A

---
## Section 2: Code

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math
from scipy.linalg import solve_banded

### Problem 1: (Circuit Analysis)


Consider a long chain of resistors wired up like this:

<img src="reschain.png" alt="Alt text" width=95%>

All the resistors have the same resistance~$R$.  The power rail at the top is at voltage~$V_+=5$V.  The problem is to find the voltages $V_1...V_N$ at the internal points in the circuit.

1. **Using Ohm's law and the Kirchhoff current law**, which says that the total net current flow out of (or into) any junction in a circuit must be zero, show that the voltages $V_1\ldots V_N$ satisfy the equations

$$3V_1 - V_2 - V_3 = V_+ $$
$$-V_1 + 4V_2 - V_3 - V_4 = V_+ $$
      
$$-V_{i-2} - V_{i-1} + 4V_i - V_{i+1} - V_{i+2} = 0 $$

$$-V_{N-3} - V_{N-2} + 4V_{N-1} - V_N = V_- = 0 $$
$$-V_{N-2} - V_{N-1} + 3V_N = V_- = 0 $$

Note: The assignment of $V_-$ is to make the equations symmetric

2. Express these equations in vector form~$A\vec{v} = \vec{w}$ and find the values of the matrix~$A$ and the vector~$\vec{w}$.

3. Write a program to solve for the values of the~$V_i$ when there are $N=6$ internal junctions with unknown voltages.  (Hint: All the values of $V_i$ should lie between zero and $5$V.  If they don't, something is wrong.)

4. Now repeat your calculation for the case where there are $N=10\,000$ internal junctions.  This part is not possible using standard tools like the *solve* function.  You need to make use of the fact that the matrix~$\vec{A}$ is banded.  

In [2]:
#for N=6
A_6 = np.array([
    [3,-1,-1,0,0,0],
    [-1,4,-1,-1,0,0],
    [-1,-1,4,-1,-1,0],
    [0,-1,-1,4,-1,-1],
    [0,0,-1,-1,4,-1,],
    [0,0,0,-1,-1,3]])
w_6 = np.zeros(6)
w_6[0]=5
w_6[1]=5
v_6= np.linalg.solve(A_6,w_6)
v_6

array([3.7254902 , 3.43137255, 2.74509804, 2.25490196, 1.56862745,
       1.2745098 ])

In [3]:
v_6 = np.dot(np.linalg.inv(A_6), w_6)
v_6

array([3.7254902 , 3.43137255, 2.74509804, 2.25490196, 1.56862745,
       1.2745098 ])

In [4]:
#for large N
def build_matrices (N,V_plus=5):
    A= np.zeros((N,N)) #builds an NxN matrix
    
    #main diagonal elements
    for i in range(N):
        A[i,i] = 3 if i in (0,N-1) else 4

    #off diagonals
    for i in range(N):
        if i+1<N:
            A[i,i+1] = -1
            A[i+1,i] = -1
        if i+2<N:
            A[i,i+2] =-1
            A[i+2,i] =-1

    #creating w matrix (first 2 elements are 5)
    w=np.zeros(N,)
    w [0] =V_plus
    if N>1:
        w [1] = V_plus
    return A, w

In [5]:
A_check, w_check = build_matrices(6)
v_check = np.linalg.solve (A_check, w_check)
v_check

array([3.7254902 , 3.43137255, 2.74509804, 2.25490196, 1.56862745,
       1.2745098 ])

In [6]:
v_check = np.dot(np.linalg.inv(A_check),w_check)
v_check

array([3.7254902 , 3.43137255, 2.74509804, 2.25490196, 1.56862745,
       1.2745098 ])

In [7]:
A_N, w_N = build_matrices(10000)
v_N = np.linalg.solve(A_N,w_N)
v_N

array([4.99888228e+00, 4.99861842e+00, 4.99802841e+00, ...,
       1.97158611e-03, 1.38158071e-03, 1.11772227e-03])

In [8]:
v_N = np.dot(np.linalg.inv(A_N),w_N)
v_N

array([4.99888228e+00, 4.99861842e+00, 4.99802841e+00, ...,
       1.97158611e-03, 1.38158071e-03, 1.11772227e-03])

In [9]:
def better_build_matrix (N, V_plus=5):

    A = np.zeros((N, N))

    np.fill_diagonal(A, 4) #fills the main diagonal with 4s

    #overwrites 4s and 0s with -1
    if N > 1: #N=1 has no main diagonal
        np.fill_diagonal(A[:, 1:], -1)  
        np.fill_diagonal(A[1:, :], -1)
    if N > 2: #N=2 is a 2x2 matrix and does not have elements 2 elements apart
        np.fill_diagonal(A[:, 2:], -1)  
        np.fill_diagonal(A[2:, :], -1)  

    #overwrites corners with 3
    A[0, 0] = 3
    if N > 1:
        A[-1, -1] = 3
    
    w = np.zeros(N,)
    w[0] = V_plus
    if N>1:
        w[1] = V_plus
    return A,w

In [10]:
A_N, w_N = better_build_matrix(10000)
v_N = np.linalg.solve(A_N,w_N)
v_N

array([4.99888228e+00, 4.99861842e+00, 4.99802841e+00, ...,
       1.97158611e-03, 1.38158071e-03, 1.11772227e-03])

In [11]:
v_N = np.dot(np.linalg.inv(A_N),w_N)
v_N

array([4.99888228e+00, 4.99861842e+00, 4.99802841e+00, ...,
       1.97158611e-03, 1.38158071e-03, 1.11772227e-03])

#### Problem 1 code summary

First, I hard coded the coefficient matrix for N=6 then used the function np.linalg.solve to get matrix v. I created the build_matrices function to recreate the hardcoded N=6 matrix. The function first adds 3 at the ends of the middle diagonal and adds -1 to the off-diagonal bands. Once the result was consistent with the hardcoded N=6 matrix it was applied to N=10,000 and matrix v was solved using np.linalg.solve, np.dot and np.linalg.inv.The function np.fill_diagonal was used for the optimized matrix builder to write elements in bands without any for loops.

### Problem 2 (Statics)


Solve the following matrix problems. Set-up the linear equations that describe the problem then solve them 

Part 1: Suppose you have a hanging mass M supported by two ropes (angled by $\alpha$ and $\beta$ with respect to the ceiling). Make a function that calculates the tension on each of the ropes. Put your function in a python file and call it here in the notebook. Check that the answer by computer matches your own computations.


<del> Part 2: 
Masses $m_1, m_2, m_3$, and $m_4$ lie on a $2.00$ [m] beam of negligible mass. They are located $0.20, 0.70, 1.10$, and $1.40$ [m] away from the left end of the beam. Determine the masses $m_1, m_2, m_3$, and $m_4$, given the following constraints: 

- The total mass is $8.00$ [kg]
- If a pivot is placed halfway, the beam will balance if a $1.05$ [kg] mass is placed on the right-end of the beam
- If a pivot is placed $1.20$ [m] away from the right end, then the beam would balance if a $550$ [g] mass is placed on the left end of the beam.
- If the system is split midway, the total mass on the left is $1.00$ [kg] heavier than the total mass on the right
</del>

In [12]:
import Problem2_part1 as p1
#input angles in degree form
p1.get_Tensions(M=20, alpha =90, beta = 90, g = 9.18 )

(91.8, 91.8)

#### Problem 2 code summary

The code calls the python program Problem2_part1.py which contains a function called get_Tensions. The program uses equations derived from the force equilibrium condition with coefficient matrix A = [$-cos(\alpha), cos(\beta)$],[$sin(\alpha), sin(\beta)$] and b matrix = [0, M*g]. It solves the tensions using np.linalg.solve.

---

## Section 3 (Notes)

### Some codes for solving matrix problems

* solving matrix equations ($\bf{A}\vec{x} = \vec{b}$) -> `np.linalg.solve`
* LU-decomposition ($\bf{A} = \bf{L}\bf{U}$) -> `scipy.linalg.lu`
* matrix inversion ($\bf{A}^{-1}$) -> `np.linalg.inv`
* QR-decomposition ($\bf{A} = \bf{Q}\bf{R}$) -> `scipy.linalg.qr`
* eigenvalue problem ($\bf{A}\vec{x} = \lambda \vec{x}$) -> `np.linalg.eig`

Solving matrix problems by themselves isn't difficult. There is an abundance of linear algebra libraries that are ready to use: from the battle-tested ones (the classic [BLAS](https://www.netlib.org/blas/) and [LAPACK](https://www.netlib.org/lapack/)), HPC tools for large problems ([ScaLAPACK](https://netlib.org/scalapack/)), to hardware-accelerated ones often with GPUs ([cuBLAS](https://docs.nvidia.com/cuda/cublas/index.html) and [cuSolver](https://docs.nvidia.com/cuda/cusolver/index.html) for NVIDIA). 

The hardest part is actually composing the matrix. How do you transform a physical problem into a linear algebra problem?

* Circuit: https://personal.math.vt.edu/embree/cmda3606/chapter2.pdf
* Embree [main notes](https://personal.math.vt.edu/embree/cmda3606notes.pdf) + [lab manual](https://personal.math.vt.edu/embree/labman.pdf)

#### Sample Usage (Testing different codes)

In [13]:
A = np.array([[1,0,2],[-2,-1,3],[0,3,5]])
B = np.array([[-2],[1],[-3]])


In [14]:
sigma_z = [[1,0],[0,-1]]
sigma_y = [[0,-1j],[1j,0]]
sigma_x = [[0,1],[1,0]]

In [15]:
ans1 = np.linalg.solve(sigma_x, sigma_y)
ans2 = np.linalg.inv(A) @ B
ans1, ans2

(array([[0.+1.j, 0.+0.j],
        [0.+0.j, 0.-1.j]]),
 array([[-1.07692308],
        [-0.23076923],
        [-0.46153846]]))

In [16]:
from scipy.linalg import *
p, l, u = lu(A)
p, l, u

(array([[0., 0., 1.],
        [1., 0., 0.],
        [0., 1., 0.]]),
 array([[ 1.        ,  0.        ,  0.        ],
        [-0.        ,  1.        ,  0.        ],
        [-0.5       , -0.16666667,  1.        ]]),
 array([[-2.        , -1.        ,  3.        ],
        [ 0.        ,  3.        ,  5.        ],
        [ 0.        ,  0.        ,  4.33333333]]))

### Importing functions from a python file

In [17]:
import samplePackage as sP

In [18]:
testMat = sP.createZerosArray(4,5)

In [19]:
testMat[1] = [1,2,3,4,5]

In [20]:
testMat

array([[0., 0., 0., 0., 0.],
       [1., 2., 3., 4., 5.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]])

### np.roll

In [21]:
## Useful tips. Not required to solve the problems below!

# Suppose you have a Numpy array
x = np.zeros((4,4))
x[1,2] = 5
for row in x:
    print(row)

#rint(x[0,1:3])


# I can move all values to the left/right using roll
x_roll = np.roll(x, 1) # change 1 to -1, what happens? 
#x_roll2 = np.roll(x, -1)
print(x_roll)

x_roll2 = np.roll(x, 2)
print(x_roll2)

[0. 0. 0. 0.]
[0. 0. 5. 0.]
[0. 0. 0. 0.]
[0. 0. 0. 0.]
[[0. 0. 0. 0.]
 [0. 0. 0. 5.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [5. 0. 0. 0.]
 [0. 0. 0. 0.]]
